# 🦠 Proyecto COVID-19 en Caldas: Guía Interactiva y Defensa Oral

Este notebook es una versión interactiva de tu proyecto basado en agentes. Aquí podrás ejecutar el código paso a paso y leer la justificación metodológica de cada decisión, ideal para estudiar antes de tu presentación.

### 🛠️ ¿Por qué esta arquitectura?
Dividir el problema en "Skills" (Data Wrangling, EDA, Modelado) hace que el código sea:
1. **Fácil de depurar:** Los errores se aíslan en sus respectivos pasos.
2. **Escalable:** Permite agregar nuevas fases sin dañar las anteriores.
3. **Lógico:** Simula el flujo real de trabajo de un científico de datos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
import warnings

warnings.filterwarnings('ignore')
print("Librerías importadas correctamente.")

## 📥 Paso 0: Carga de Datos Crudos
Empezamos cargando el dataset tal cual como viene del mundo real.

In [ ]:
# Asegúrate de tener el archivo en la misma carpeta que este notebook
try:
    df_crudo = pd.read_csv('datos_crudos/Casos_positivos_de_COVID-19_en_Caldas.csv', sep=';', encoding='utf-8', on_bad_lines='skip')
    print("Dataset cargado. Tamaño:", df_crudo.shape)
    display(df_crudo.head(3))
except FileNotFoundError:
    print("⚠️ Error: No se encontró el archivo 'datos_crudos/Casos_positivos_de_COVID-19_en_Caldas.csv'. Por favor descárgalo y colócalo aquí.")

--- 
## 🧹 Skill 1: Data Wrangling (Limpieza)

**🎤 Argumentos para tu Defensa Oral:**

*   **Curación de Columnas:** El dataset original tenía problemas de codificación (`—` en lugar de acentos). Se solucionó creando un diccionario de mapeo, demostrando rigor técnico.
*   **Feature Selection:** Eliminamos identificadores (`ID de caso`) y fechas descriptivas. Los algoritmos de Machine Learning aprenden de características inherentes (Edad, Sexo), no de folios burocráticos.
*   **Feature Engineering (Target_Gravedad):** Transformamos múltiples estados ambiguos en un problema binario claro: `0` (Leve) y `1` (Grave/Fallecido). Esto simplifica matemáticamente el aprendizaje del algoritmo.

In [ ]:
df_clean = df_crudo.copy() if 'df_crudo' in locals() else pd.DataFrame()

if not df_clean.empty:
    # 1. Renombrar columnas con mala codificación
    columnas_renombradas = {
        'Fecha de notificaci—n': 'Fecha de notificacion',
        'C—digo DIVIPOLA departamento': 'Codigo departamento',
        'C—digo DIVIPOLA municipio': 'Codigo municipio',
        'Ubicaci—n del caso': 'Ubicacion del caso',
        'C—digo ISO del pa’s': 'Codigo ISO pais',
        'Nombre del pa’s': 'Nombre pais',
        'Fecha de inicio de s’ntomas': 'Fecha de inicio de sintomas',
        'Fecha de diagnostico': 'Fecha de diagnostico', 
        'Fecha de diagn—stico': 'Fecha de diagnostico',
        'Fecha de recuperaci—n': 'Fecha de recuperacion',
        'Tipo de recuperaci—n': 'Tipo de recuperacion',
        'Pertenencia Žtnica': 'Pertenencia etnica',
        'Nombre del grupo Žtnico': 'Nombre grupo etnico'
    }
    df_clean.rename(columns=columnas_renombradas, inplace=True)

    # 2. Eliminar ruido y columnas que no aportan predicción
    columnas_a_eliminar = [
        'fecha reporte web', 'ID de caso', 'Fecha de notificacion', 
        'Codigo departamento', 'Nombre departamento', 'Codigo municipio', 
        'Codigo ISO pais', 'Nombre pais', 'Fecha de inicio de sintomas', 
        'Fecha de diagnostico', 'Fecha de recuperacion', 'Tipo de recuperacion', 
        'Fecha de muerte', 'Pertenencia etnica', 'Nombre grupo etnico', 
        'Unidad de medida de edad', 'Recuperado'
    ]
    df_clean.drop(columns=columnas_a_eliminar, inplace=True, errors='ignore')

    # 3. Limpieza de Nulos
    df_clean.dropna(subset=['Estado', 'Ubicacion del caso', 'Tipo de contagio'], inplace=True)

    # 4. Target binario
    def clasificar_gravedad(estado):
        estado_str = str(estado).lower()
        if 'leve' in estado_str or 'asintomático' in estado_str:
            return 0  # Riesgo Bajo
        else:
            return 1  # Riesgo Alto
            
    df_clean['Target_Gravedad'] = df_clean['Estado'].apply(clasificar_gravedad)
    df_clean.drop(columns=['Estado', 'Ubicacion del caso'], inplace=True)

    # 5. Label Encoding (Categorías a Números)
    le = LabelEncoder()
    df_clean['Sexo_encoded'] = le.fit_transform(df_clean['Sexo'])
    df_clean['Tipo_contagio_encoded'] = le.fit_transform(df_clean['Tipo de contagio'])
    df_clean['Municipio_encoded'] = le.fit_transform(df_clean['Nombre municipio'])
    
    df_clean.drop(columns=['Sexo', 'Tipo de contagio', 'Nombre municipio'], inplace=True)

    print("Data Wrangling Completado. Variables listas:")
    display(df_clean.head(3))

--- 
## 📊 Skill 2: Exploratory Data Analysis (EDA)

**🎤 Argumentos para tu Defensa Oral:**

*   **Evidencia Tangible:** Generamos los gráficos y los guardamos para evidenciar visualmente los hallazgos en la carpeta `evidencias_eda/`.
*   **Tablas de Frecuencia Rigurosas:** Revelan si existe un desbalance de clases severo (ej. si casi todos los casos son leves). Es crítico saber esto antes de modelar.
*   **Uso de Cuartiles (Boxplot):** Es la representación óptima para entender visualmente dónde se concentra la población de mayor riesgo según su Edad.
*   **Matriz de Correlación:** Nuestro puente lógico hacia el modelado. Justificamos la selección de variables demostrando numéricamente cuáles influyen más sobre el `Target_Gravedad`.

In [ ]:
if not df_clean.empty:
    os.makedirs('graficos', exist_ok=True)
    
    print("--- Frecuencias del Target ---")
    frec_abs = df_clean['Target_Gravedad'].value_counts()
    frec_rel = df_clean['Target_Gravedad'].value_counts(normalize=True) * 100
    df_frec = pd.DataFrame({'Absoluta': frec_abs, 'Relativa (%)': frec_rel.round(2)})
    df_frec.index = ['Leve (0)', 'Grave/Fallecido (1)']
    display(df_frec)

    sns.set_theme(style="whitegrid")
    
    # Mostrar Boxplot aquí interactivo
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df_clean, x='Target_Gravedad', y='Edad', palette='Set2')
    plt.title('Distribución de Edad según Gravedad')
    plt.xticks(ticks=[0, 1], labels=['Leve', 'Grave'])
    plt.show()
    
    # Matriz de Correlación
    plt.figure(figsize=(7, 5))
    sns.heatmap(df_clean.corr(), annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
    plt.title('Matriz de Correlación')
    plt.show()

--- 
## 🧠 Skill 3: Modelado Predictivo

**🎤 Argumentos para tu Defensa Oral:**

*   **La partición `stratify=y`:** Garantiza que el subconjunto de examen (Test) tenga la misma proporción exacta de casos graves/leves que el entrenamiento, haciendo la evaluación imparcial.
*   **El parámetro `class_weight='balanced'`:** Al haber una inmensa mayoría de casos leves, el algoritmo podría volverse perezoso y predecir siempre "Leve". Con este parámetro, obligamos matemáticamente al modelo a penalizar duramente sus errores cuando falla detectando un caso grave.
*   **Métrica F1-Score vs Accuracy:** La exactitud (Accuracy) es engañosa; si el 90% es leve, un modelo que siempre predice "Leve" tendría 90% de Accuracy pero sería un peligro médico. El F1-Score cruza precisión con sensibilidad (recall), siendo el juez implacable para datos médicos desbalanceados.

In [ ]:
if not df_clean.empty:
    # 1. Separación de datos
    X = df_clean.drop(columns=['Target_Gravedad'])
    y = df_clean['Target_Gravedad']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # 2. Algoritmos balanceados
    modelo_rl = LogisticRegression(class_weight='balanced', random_state=42)
    modelo_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    
    # 3. Entrenamiento
    modelo_rl.fit(X_train, y_train)
    modelo_rf.fit(X_train, y_train)
    
    # 4. Predicción
    pred_rl = modelo_rl.predict(X_test)
    pred_rf = modelo_rf.predict(X_test)
    
    # 5. Evaluación
    f1_rl = f1_score(y_test, pred_rl, average='weighted')
    f1_rf = f1_score(y_test, pred_rf, average='weighted')
    
    print("--- Puntajes F1-Score ---")
    print(f"Regresión Logística: {f1_rl:.4f}")
    print(f"Random Forest      : {f1_rf:.4f}")
    
    # 6. Decisión
    if f1_rf >= f1_rl:
        mejor_modelo, final_pred = modelo_rf, pred_rf
        print("\n🏆 Ganador: Random Forest")
    else:
        mejor_modelo, final_pred = modelo_rl, pred_rl
        print("\n🏆 Ganador: Regresión Logística")

    print("\n--- Reporte Final ---")
    print(classification_report(y_test, final_pred, target_names=['Leve (0)', 'Grave (1)']))